In [ ]:
import sys
import os
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from IPython.display import Video, display, HTML
import pandas as pd
from typing import Dict, List
import time

# Import our custom modules
from src.main_pipeline import PersonReIDPipeline
from src.detection.infer import YOLODetector
from src.tracking.huffman_tracker import PersonTracker
from src.reid.osnet_reid import OSNetReID
from src.index.hnsw_index import HNSWIndex
from src.fusion.fusion import IDFusion

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("✓ All imports successful")


In [ ]:
# Check if video exists
video_path = "../data/raw/video.mp4"

if not os.path.exists(video_path):
    print(f"❌ Video not found at {video_path}")
    print("Please place your test video at data/raw/video.mp4")
else:
    print(f"✓ Video found at {video_path}")
    
    # Get video info
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    
    print(f"Video properties:")
    print(f"  - Resolution: {width}x{height}")
    print(f"  - FPS: {fps}")
    print(f"  - Frames: {frame_count}")
    print(f"  - Duration: {duration:.2f} seconds")
    
    cap.release()
    
    # Display first frame
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    if ret:
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title("First Frame of Input Video")
        plt.axis('off')
        plt.show()
    cap.release()


In [ ]:
# Create output directory
output_dir = "../output/demo_results"
os.makedirs(output_dir, exist_ok=True)

# Initialize complete pipeline
print("Initializing complete pipeline...")
pipeline = PersonReIDPipeline(
    yolo_model_path="yolov8n.pt",
    reid_model_name="osnet_x1_0",
    hnsw_dim=2048,
    hnsw_max_elements=10000,
    output_dir=output_dir,
    save_crops=True,
    save_video=True,
    device="auto"
)
print("✓ Complete pipeline initialized")


In [ ]:
# Process video (limit to first 300 frames for demo)
if os.path.exists(video_path):
    print("Processing video through complete pipeline...")
    print("Note: Processing limited to first 300 frames for demo")
    
    start_time = time.time()
    
    stats = pipeline.process_video(
        video_path=video_path,
        max_frames=300  # Limit for demo
    )
    
    processing_time = time.time() - start_time
    
    print(f"\n✓ Processing completed in {processing_time:.2f} seconds")
    print(f"✓ Results saved to {output_dir}")
else:
    print("❌ Video not found. Please place your test video at data/raw/video.mp4")
    stats = None


In [ ]:
if stats:
    print("=" * 60)
    print("PIPELINE PROCESSING RESULTS")
    print("=" * 60)
    
    # Basic statistics
    print(f"Total frames processed: {stats['total_frames']}")
    print(f"Total detections: {stats['total_detections']}")
    print(f"Average detections per frame: {stats['avg_detections_per_frame']:.2f}")
    print(f"Maximum tracks: {stats['max_tracks']}")
    print(f"Average FPS: {stats['avg_fps']:.2f}")
    
    # Fusion statistics
    fusion_stats = stats['fusion_stats']
    print(f"\nID Fusion Results:")
    print(f"  - Active persons: {fusion_stats['active_persons']}")
    print(f"  - Total identities created: {fusion_stats['next_global_id']}")
    print(f"  - Total fusions: {fusion_stats['total_fusions']}")
    print(f"  - ReID matches: {fusion_stats['reid_matches']}")
    print(f"  - Tracking matches: {fusion_stats['tracking_matches']}")
    print(f"  - New identities: {fusion_stats['new_identities']}")
    
    # Processing time breakdown
    print(f"\nProcessing Time Breakdown:")
    for stage, times in stats['processing_times'].items():
        print(f"  - {stage.capitalize()}: {times['mean']*1000:.2f}ms ± {times['std']*1000:.2f}ms")
    
    print("=" * 60)


In [ ]:
if stats and 'processing_times' in stats:
    # Plot processing time breakdown
    stages = []
    mean_times = []
    std_times = []
    
    for stage, times in stats['processing_times'].items():
        if stage != 'total':  # Skip total for breakdown
            stages.append(stage.capitalize())
            mean_times.append(times['mean'] * 1000)  # Convert to ms
            std_times.append(times['std'] * 1000)
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    bars = plt.bar(stages, mean_times, yerr=std_times, capsize=5, alpha=0.7)
    plt.ylabel('Processing Time (ms)')
    plt.title('Processing Time by Stage')
    plt.xticks(rotation=45)
    
    # Add value labels on bars
    for bar, mean_time in zip(bars, mean_times):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{mean_time:.1f}ms', ha='center', va='bottom')
    
    # Pie chart of time distribution
    plt.subplot(1, 2, 2)
    plt.pie(mean_times, labels=stages, autopct='%1.1f%%', startangle=90)
    plt.title('Processing Time Distribution')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Show person galleries
if stats and stats['fusion_stats']['active_persons'] > 0:
    print("Person Galleries:")
    
    # Get all person IDs
    max_persons = min(5, stats['fusion_stats']['next_global_id'])  # Show max 5 persons
    
    for person_id in range(max_persons):
        crop_paths = pipeline.get_person_gallery(person_id, max_crops=8)
        
        if crop_paths:
            print(f"\nPerson {person_id}: {len(crop_paths)} crops")
            
            # Display crops
            plt.figure(figsize=(15, 3))
            
            for i, crop_path in enumerate(crop_paths[:8]):
                if os.path.exists(crop_path):
                    crop = cv2.imread(crop_path)
                    if crop is not None:
                        plt.subplot(1, 8, i+1)
                        plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                        plt.title(f"Frame {Path(crop_path).stem.split('_')[1]}")
                        plt.axis('off')
            
            plt.suptitle(f"Person {person_id} Gallery", fontsize=16)
            plt.tight_layout()
            plt.show()
else:
    print("No person galleries available")


In [ ]:
print("=" * 80)
print("PERSON RE-IDENTIFICATION SYSTEM DEMO SUMMARY")
print("=" * 80)

if stats:
    print(f"✓ Successfully processed {stats['total_frames']} frames")
    print(f"✓ Detected {stats['total_detections']} person instances")
    print(f"✓ Tracked {stats['max_tracks']} simultaneous persons")
    print(f"✓ Identified {stats['fusion_stats']['next_global_id']} unique persons")
    print(f"✓ Achieved {stats['avg_fps']:.2f} FPS processing speed")
    
    print("\nSystem Components Performance:")
    for stage, times in stats['processing_times'].items():
        if stage != 'total':
            print(f"  - {stage.capitalize()}: {times['mean']*1000:.1f}ms average")
    
    print("\nOutput Files Generated:")
    output_path = Path(output_dir)
    print(f"  - Annotated video: {output_path / 'videos'}")
    print(f"  - Person crops: {output_path / 'crops'}")
    print(f"  - Processing data: {output_path / 'data'}")
    print(f"  - System logs: {output_path / 'pipeline.log'}")
    
    print("\nKey Features Demonstrated:")
    print("  ✓ Real-time person detection with YOLO")
    print("  ✓ Efficient coordinate tracking with Huffman encoding")
    print("  ✓ Robust feature extraction with OSNet")
    print("  ✓ Fast similarity search with HNSW indexing")
    print("  ✓ Intelligent ID fusion for consistent tracking")
    print("  ✓ Comprehensive result visualization and analysis")
    
else:
    print("❌ Demo could not be completed - please check video file")
    
print("\nNext Steps:")
print("  - Adjust pipeline parameters for your specific use case")
print("  - Train custom ReID models for better performance")
print("  - Implement real-time processing for live video streams")
print("  - Add database integration for persistent person storage")
print("  - Extend with face recognition for enhanced accuracy")

print("=" * 80)
